# M04B: Role-Based Prompts & Personas

Same model, different behavior. A "teacher" explains patiently. A "code reviewer" is direct and thorough.

**Topics:**
- Specialized personas (teacher, code reviewer, technical writer)
- PersonaConversation class
- Persona consistency across multi-turn chats

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


# Display helper for long outputs
def truncate_response(text, max_length=1200):
    """Truncate text for readability."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"


print("✅ Ready!")

---

## 🎭 From Role to Persona

A **role** is identity ("You are a senior software engineer.").

A **persona** adds behaviors and communication style on top of that identity.

**Persona = Role + Behaviors + Communication Style**

In [ ]:
code_to_review = """
def calculate_area(radius):
    import math
    return 3.14 * radius * radius
"""

print("👤 ROLE ONLY")
print("="*60)
response_role = client.responses.create(
    model=MODEL,
    input=f"Review this code:\n{code_to_review}",
    instructions="You are a senior software engineer."
)
print(truncate_response(response_role.output_text, max_length=800))

print("\n🎭 PERSONA A: Strict Reviewer")
print("="*60)
response_strict = client.responses.create(
    model=MODEL,
    input=f"Review this code:\n{code_to_review}",
    instructions=(
        "You are a senior software engineer.\n"
        "Focus on: correctness, performance, coding standards.\n"
        "Be direct and concise."
    )
)
print(truncate_response(response_strict.output_text, max_length=800))

print("\n🎭 PERSONA B: Encouraging Mentor")
print("="*60)
response_mentor = client.responses.create(
    model=MODEL,
    input=f"Review this code:\n{code_to_review}",
    instructions=(
        "You are a senior software engineer mentoring "
        "a junior developer.\n"
        "Focus on: building understanding, explaining "
        "the why behind each suggestion.\n"
        "Be encouraging and patient. Use simple language."
    )
)
print(truncate_response(response_mentor.output_text, max_length=800))

print("="*60)

### 💡 Key Observation

Same role, same code — the strict reviewer leads with problems, the mentor leads with encouragement and explains the why behind each fix.

---

## 📋 What Makes an Effective Persona?

A good persona defines:
1. **Role** — Who they are
2. **Behaviors** — What they focus on
3. **Communication style** — How they talk

**❌ Too Vague:**
```
"You are helpful."
```

**❌ Contradictory:**
```
"You are formal but casual and serious but funny."
```

**✅ Clear and Specific:**
```
"You are a senior software engineer.
Focus on: security, performance, maintainability.
Be direct but constructive. Be concise."
```

---

## 📚 Building a Persona Library

Create reusable personas for common tasks.

In [ ]:
PERSONAS = {
    "code_reviewer": """You are a senior software engineer.
Focus on: security, performance, maintainability.
Be direct but constructive. Be concise.""",

    "teacher": """You are a patient teacher.
Focus on: clear explanations, simple examples, building understanding.
Be encouraging and supportive. Use analogies. Be concise.""",

    "technical_writer": """You are a technical documentation writer.
Focus on: clarity, completeness, accuracy.
Be precise and professional. Be concise.""",

    "customer_support": """You are a customer support specialist.
Focus on: understanding issues, providing solutions, being helpful.
Be empathetic, patient, and solution-focused. Be concise."""
}

custom_personas = {}


def get_persona(name):
    """Get persona instructions by name (case-insensitive)."""
    name = name.lower()
    if name in PERSONAS:
        return PERSONAS[name]
    if name in custom_personas:
        return custom_personas[name]
    raise ValueError(f"Persona '{name}' not found")


def list_personas():
    """List all available persona names."""
    return list(PERSONAS.keys()) + list(custom_personas.keys())


def add_persona(name, instructions):
    """Add a custom persona."""
    custom_personas[name.lower()] = instructions
    print(f"✅ Added custom persona: {name}")


# --------------------------------------------------------------
print("✅ Persona library ready!")
print("📚 AVAILABLE PERSONAS")
print("="*60)
for persona in list_personas():
    print(f"  - {persona}")

---

## 🧪 Testing Different Personas

In [ ]:
question = "Explain what a REST API is"

print("👨‍💻 CODE REVIEWER")
print("="*60)
response = client.responses.create(
    model=MODEL,
    input=question,
    instructions=get_persona("code_reviewer")
)
print(truncate_response(response.output_text, max_length=800))

print("\n👨‍🏫 TEACHER")
print("="*60)
response = client.responses.create(
    model=MODEL,
    input=question,
    instructions=get_persona("teacher")
)
print(truncate_response(response.output_text, max_length=800))

print("\n📝 TECHNICAL WRITER")
print("="*60)
response = client.responses.create(
    model=MODEL,
    input=question,
    instructions=get_persona("technical_writer")
)
print(truncate_response(response.output_text, max_length=800))

print("="*60)

### 💡 Key Observation

Same question, three personas, three different approaches. The persona library makes it easy to switch between them.

---

## 🔗 Maintaining Persona Consistency Across Turns

This builds on `ConversationManager` from M04A and adds integration with the persona library.

In [ ]:
class PersonaConversation:
    """Manage multi-turn conversations with a persona."""
    
    def __init__(self, persona_name):
        self.persona_name = persona_name
        self.instructions = get_persona(persona_name)
        self.last_response_id = None
        self.transcript = []
    
    def send(self, message):
        """Send message and return response."""
        try:
            kwargs = {
                "model": MODEL,
                "input": message,
                "instructions": self.instructions
            }
            
            if self.last_response_id:
                kwargs["previous_response_id"] = self.last_response_id
            
            response = client.responses.create(**kwargs)
            
            response_text = response.output_text.strip()
            self.last_response_id = response.id
            
            self.transcript.append({
                "user": message,
                "assistant": response_text
            })
            
            return response_text
            
        except Exception as e:
            print(f"❌ API Error: {e}")
            return f"Error: {str(e)}"
    
    def show_transcript(self):
        """Display conversation transcript."""
        print(f"\n📜 CONVERSATION TRANSCRIPT ({self.persona_name})")
        print("="*60)
        
        for i, turn in enumerate(self.transcript, 1):
            print(f"\nTurn {i}:")
            print(f"👤 User: {turn['user']}")
            print(f"🤖 AI: {truncate_response(turn['assistant'])}")
        
        print("="*60)
    
    def reset(self):
        """Start a new conversation."""
        self.last_response_id = None
        self.transcript = []
        print(f"✅ Conversation reset ({self.persona_name})")


# --------------------------------------------------------------
print("✅ PersonaConversation ready!")

---

## 🎯 Demo: Multi-Turn Persona Consistency

In [ ]:
conversation = PersonaConversation("teacher")

print("🎓 TEACHER PERSONA DEMO")
print("="*60)

response1 = conversation.send("What is a for loop?")
print(f"\nTurn 1:")
print(f"👤 User: What is a for loop?")
print(f"🤖 Teacher: {truncate_response(response1)}")

response2 = conversation.send("Can you show me an example?")
print(f"\nTurn 2:")
print(f"👤 User: Can you show me an example?")
print(f"🤖 Teacher: {truncate_response(response2)}")

response3 = conversation.send("What's the difference from a while loop?")
print(f"\nTurn 3:")
print(f"👤 User: What's the difference from a while loop?")
print(f"🤖 Teacher: {truncate_response(response3)}")

conversation.show_transcript()

### 💡 Key Benefit

`PersonaConversation` pulls from the persona library — just pass a name, not the full instructions string.

---

### 💪 Your Turn: Build Your Custom Persona

Create a persona for your specific use case using the 3-element structure: Identity, Behaviors, Style.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Build Your Custom Persona
# --------------------------------------------------------------
# Objective: Define a persona and test it in a conversation.
#
# 1. Define your persona using the 3-element structure (role, behaviors, communication style)
# 2. Add it to the library with add_persona("name", my_persona)
# 3. Create a PersonaConversation with your persona
# 4. Test with a question and a follow-up

my_persona = """
You are a [ROLE].
Focus on: [BEHAVIORS].
Be [COMMUNICATION STYLE]. Be concise.
"""

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**🎭 Effective Personas:**
- Clear identity (who they are)
- Specific behaviors (what they focus on)
- Communication style (how they talk)

**🔗 Keeping Personas Consistent Across Turns:**
- For consistent behavior, pass instructions on every turn
- Use `PersonaConversation` to handle state automatically
- Always link `previous_response_id` for context

**📋 When to Use Personas:**
- Domain-specific tasks (coding, law, medicine)
- Multi-turn conversations (chatbots, tutors)
- **The Flow:** Choose persona → Create `PersonaConversation` → Instructions auto-applied every turn

---

### 📍 Next Step

**M04C: Few-Shot Learning Advanced** — Complex tasks, combining few-shot with personas, and dynamic example selection.

---

## 🔧 Troubleshooting

**Persona not working as expected?**
- Make instructions more specific
- Add concrete behavioral guidelines
- Check for contradictions

**Persona inconsistent across turns?**
- If you don't pass instructions each turn, behavior may drift
- Pass persona instructions every turn for consistent behavior
- Use `PersonaConversation` for automatic consistency

**Persona too generic or too specific?**
- Too generic: Add specific expertise and behaviors
- Too specific: Remove overly detailed constraints
- Test and iterate to find balance


**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output


---